# 캐글 대회

In [374]:
# 필요한 모듈 임포트
import os, hds
import pandas as pd
import numpy as np
from plt_rcs import *

In [375]:
os.getcwd()

'/Users/taehyunan/Desktop/Repo/SeSAC/Study/sesac_ml_dl_study_repo/kaggle_data_2/data'

In [376]:
os.chdir('../data')

In [377]:
sorted(os.listdir())

['base_lgbm_model.csv', 'sample_submission.csv', 'test.csv', 'train.csv']

In [378]:
# 데이터셋 불러오기
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

## 데이터 조회

In [379]:
train.head()

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Target,Failure Type,Num ID
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,No Failure,14860
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,No Failure,47181
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,No Failure,47182
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,No Failure,47183
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,No Failure,47184


In [380]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 11 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   UDI                      10000 non-null  int64  
 1   Product ID               10000 non-null  object 
 2   Type                     10000 non-null  object 
 3   Air temperature [K]      10000 non-null  float64
 4   Process temperature [K]  10000 non-null  float64
 5   Rotational speed [rpm]   10000 non-null  int64  
 6   Torque [Nm]              10000 non-null  float64
 7   Tool wear [min]          10000 non-null  int64  
 8   Target                   10000 non-null  int64  
 9   Failure Type             10000 non-null  object 
 10  Num ID                   10000 non-null  int64  
dtypes: float64(3), int64(5), object(3)
memory usage: 859.5+ KB


In [381]:
train.describe().round(3)

,UDI,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Target,Num ID
count,10000.000,10000.000,10000.000,10000.000,10000.000,10000.000,10000.000,10000.000
mean,5000.500,300.005,310.006,1538.776,39.987,107.951,0.034,40711.266
std,2886.896,2.000,1.484,179.284,9.969,63.654,0.181,14870.161
min,1.000,295.300,305.700,1168.000,3.800,0.000,0.000,14860.000
25%,2500.750,298.300,308.800,1423.000,33.200,53.000,0.000,23214.750
50%,5000.500,300.100,310.100,1503.000,40.100,108.000,0.000,48861.500
75%,7500.250,301.500,311.100,1612.000,46.800,162.000,0.000,53001.500
max,10000.000,304.500,313.800,2886.000,76.600,253.000,1.000,57174.000


In [382]:
train.describe(include=object)

,Product ID,Type,Failure Type
count,10000,10000,10000
unique,10000,3,6
top,M14860,L,No Failure
freq,1,6000,9652


In [383]:
train['Failure Type'].value_counts()

Failure Type
No Failure                  9652
Heat Dissipation Failure     112
Power Failure                 95
Overstrain Failure            78
Tool Wear Failure             45
Random Failures               18
Name: count, dtype: int64

In [384]:
test.head()

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Failure Type
0,10001,M24860,M,300.5,309.5,1451,47.8,93,No Failure
1,10002,M24861,M,301.4,311.1,1697,35.6,160,No Failure
2,10003,H39413,H,304.0,313.1,1612,33.7,100,No Failure
3,10004,L57175,L,298.6,310.5,1276,55.2,91,No Failure
4,10005,L57176,L,299.9,310.8,1400,46.2,219,No Failure


In [385]:
test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 9 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   UDI                      3000 non-null   int64  
 1   Product ID               3000 non-null   object 
 2   Type                     3000 non-null   object 
 3   Air temperature [K]      3000 non-null   float64
 4   Process temperature [K]  3000 non-null   float64
 5   Rotational speed [rpm]   3000 non-null   int64  
 6   Torque [Nm]              3000 non-null   float64
 7   Tool wear [min]          3000 non-null   int64  
 8   Failure Type             3000 non-null   object 
dtypes: float64(3), int64(3), object(3)
memory usage: 211.1+ KB


## 오입력 처리

In [386]:
# 오입력 케이스B : Target=1 + No Failure → 0
# train.loc[train['Failure Type'].eq('No Failure') & train['Target'].eq(1), 'Target'] = 0

In [387]:
# 오입력 케이스C : Target=0 + Random Failures → 1
# train.loc[train['Failure Type'].ne('No Failure') & train['Target'].eq(0), 'Target'] = 1

## 특성 행렬과 타겟 벡터로 분리

In [388]:
train.columns

Index(['UDI', 'Product ID', 'Type', 'Air temperature [K]',
       'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]',
       'Tool wear [min]', 'Target', 'Failure Type', 'Num ID'],
      dtype='object')

In [389]:
# 타겟 벡터명 설정
yvar = 'Target'

# 훈련셋 특성 행렬 생성
X = train.drop(columns = yvar)

# 훈련셋 타겟 벡터 생성
y = train[yvar].copy()

In [390]:
# 시험셋 특성 행렬 생성
X_test = test.copy()

## 데이터 전처리

In [391]:
# UID / Failure Type 제거 및 Product ID 인덱스 설정
X = X.drop(columns=['UDI', 'Failure Type', 'Num ID']).set_index(keys='Product ID')
X_test = X_test.drop(columns=['UDI', 'Failure Type']).set_index(keys='Product ID')

In [392]:
X.columns

Index(['Type', 'Air temperature [K]', 'Process temperature [K]',
       'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]'],
      dtype='object')

In [393]:
X_test.columns

Index(['Type', 'Air temperature [K]', 'Process temperature [K]',
       'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]'],
      dtype='object')

In [394]:
# 컬럼명 변경
X.columns = ['Type', 'AirTmp', 'ProcTmp', 'RotSpd', 'Torque', 'ToolWear']
X_test.columns = ['Type', 'AirTmp', 'ProcTmp', 'RotSpd', 'Torque', 'ToolWear']

In [395]:
X.head()

,Type,AirTmp,ProcTmp,RotSpd,Torque,ToolWear
Product ID,,,,,,
M14860,M,298.1,308.6,1551,42.8,0
L47181,L,298.2,308.7,1408,46.3,3
L47182,L,298.1,308.5,1498,49.4,5
L47183,L,298.2,308.6,1433,39.5,7
L47184,L,298.2,308.7,1408,40.0,9


In [396]:
# Type 오디널 인코딩: L=0, M=1, H=2
from sklearn.preprocessing import OrdinalEncoder

oe = OrdinalEncoder(categories=[['L', 'M', 'H']])
X['Type'] = oe.fit_transform(X[['Type']])
X_test['Type'] = oe.transform(X_test[['Type']])

X.head()

,Type,AirTmp,ProcTmp,RotSpd,Torque,ToolWear
Product ID,,,,,,
M14860,1.0,298.1,308.6,1551,42.8,0
L47181,0.0,298.2,308.7,1408,46.3,3
L47182,0.0,298.1,308.5,1498,49.4,5
L47183,0.0,298.2,308.6,1433,39.5,7
L47184,0.0,298.2,308.7,1408,40.0,9


## 피처 엔지니어링

In [397]:
from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

In [398]:
# 파생변수 추가 함수
def add_features(df):
    df = df.copy()
    # 열 방출 능력(상관계수 0.88)
    df['temp_diff'] = df['ProcTmp'] - df['AirTmp']
    # 기계 출력(상관계수 -0.88)
    df['power'] = df['Torque'] * df['RotSpd']
    # 마모 상태에서의 부하
    df['wear_torque'] = df['ToolWear'] * df['Torque']
    
    return df

In [399]:
# 파생변수 추가
X = add_features(X)
X_test = add_features(X_test)

In [412]:
X_test.describe().round(3)

,Type,AirTmp,ProcTmp,RotSpd,Torque,ToolWear,temp_diff,power,wear_torque
count,3000.000,3000.000,3000.000,3000.000,3000.000,3000.000,3000.000,3000.000,3000.000
mean,0.488,300.058,310.029,1541.423,39.830,107.698,9.971,59687.741,4313.689
std,0.663,2.090,1.562,185.174,10.449,65.660,1.033,10680.351,2970.106
min,0.000,294.300,305.300,1119.000,0.200,0.000,7.200,594.200,0.000
25%,0.000,298.400,308.800,1420.000,32.600,52.000,9.200,52683.350,1870.675
50%,0.000,300.000,310.000,1514.000,39.950,105.000,9.800,59717.200,4016.800
75%,1.000,301.700,311.200,1625.250,47.100,162.000,10.900,67079.650,6236.825
max,2.000,305.500,314.500,2971.000,73.400,265.000,12.600,99897.400,14637.600


## 모델 정의

### LGBM 모델

In [400]:
model = LGBMClassifier(
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
)

### XGBoost 모델

In [401]:
# from xgboost import XGBClassifier

# model = XGBClassifier(
#     scale_pos_weight=len(y[y==0]) / len(y[y==1]),  # class_weight='balanced' 대응
#     random_state=42,
#     n_jobs=-1,
# )

### RF 모델

In [402]:
# from sklearn.ensemble import RandomForestClassifier

# model = RandomForestClassifier(
#     class_weight='balanced',
#     random_state=42,
#     n_jobs=-1,
# )

### SMOTE 모델

In [403]:
# from imblearn.over_sampling import SMOTE
# from imblearn.pipeline import Pipeline as ImbPipeline

# # SMOTE 적용 모델 (class_weight 제거)
# model_smote = LGBMClassifier(random_state=42, n_jobs=-1)

# model = ImbPipeline([
#     ('smote', SMOTE(random_state=42)),
#     ('model', model_smote),
# ])

In [404]:
# Stratified K-Fold CV
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X, y, cv=skf, scoring='roc_auc', n_jobs=-1)

print(f"CV AUC 평균: {cv_scores.mean():.5f}")
print(f"Fold별 편차: {cv_scores.std():.5f}")
print(f"Fold별 점수: {cv_scores.round(5)}")

[LightGBM] [Info] Number of positive: 272, number of negative: 7728
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001623 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1516
[LightGBM] [Info] Number of positive: 271, number of negative: 7729
[LightGBM] [Info] Number of data points in the train set: 8000, number of used features: 9
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002623 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1518
[LightGBM] [Info] Number of data points in the train set: 8000, number of used features: 9
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
[LightGBM] [Info] Number

## SHAP 피처 중요도 확인

In [405]:
# import shap

# # X_train 없으므로 전체 X 사용 (model_output='probability'로 확률 기반 SHAP)
# shap_explainer = shap.TreeExplainer(model=model, data=X, model_output='probability')
# shap_values = shap_explainer.shap_values(X=X, check_additivity=False)

# # DataFrame으로 변환
# shap_1 = pd.DataFrame(data=shap_values, columns=model.feature_names_in_)

# # 특성별 SHAP 값의 절대값 평균 확인
# shap_1.abs().mean().sort_values(ascending=False)

## 전체 훈련셋으로 재학습 및 제출 파일 생성

In [407]:
# 전체 훈련셋으로 재학습
model.fit(X, y)

# 시험셋 예측확률 생성
y_test_prob = model.predict_proba(X_test)[:, 1]

# 제출 파일 생성
submit = pd.read_csv('sample_submission.csv')
submit[yvar] = y_test_prob

key = 'default'
submit_filename = f'{key}_model.csv'
submit.to_csv(path_or_buf=submit_filename, index=False)

[LightGBM] [Info] Number of positive: 339, number of negative: 9661
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001304 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1522
[LightGBM] [Info] Number of data points in the train set: 10000, number of used features: 9
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
